In [0]:
from pyspark.sql.functions import (
    col, row_number, current_timestamp, sha2, concat_ws,
    to_date, expr
)
from pyspark.sql.window import Window

bronze_df = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_raw")

cleaned_df = (bronze_df
    .withColumnRenamed("series", "series_bk")
    .withColumnRenamed("duoarea", "duoarea_bk")
    .withColumnRenamed("product", "product_bk")
    .withColumn("period_bk", expr("try_cast(period as date)"))
    .withColumn("consumption_value", expr("try_cast(value as decimal(10,3))"))
    .select(
        "series_bk", "duoarea_bk", "period_bk", "product_bk",
        "units", "consumption_value", "source_filename", "ingestion_timestamp"
    )
)


In [0]:
w = Window.partitionBy("series_bk", "duoarea_bk", "period_bk") \
          .orderBy(col("ingestion_timestamp").desc())

deduped_df = (cleaned_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
from pyspark.sql.functions import lit

final_df = deduped_df.withColumn(
    "consumption_sk",
    sha2(concat_ws("||", 
        col("series_bk"), 
        col("duoarea_bk"), 
        col("period_bk").cast("string")
    ), 256)
).withColumn(
    "_source_system", lit("EIA_petroleum_consumption")
).withColumn(
    "_ingested_at", current_timestamp()
).withColumn(
    "_updated_at", lit(None).cast("timestamp")  
).select(
    "consumption_sk", "series_bk", "duoarea_bk", "period_bk", "product_bk",
    "units", "consumption_value", "_source_system", "_ingested_at", "_updated_at"
)

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(
    spark, "dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption"
)

(target_table.alias("t")
    .merge(final_df.alias("s"), "t.consumption_sk = s.consumption_sk")
    .whenMatchedUpdate(
        condition="t.consumption_value <> s.consumption_value",
        set={
            "consumption_value": "s.consumption_value",
            "_source_system": "s._source_system",
            "_updated_at": "current_timestamp()"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.sql("""
SELECT COUNT(*) AS row_count 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_consumption
""").show()